<a href="https://colab.research.google.com/github/joshuacvo/Joshua_INFO4670_Fall2026/blob/main/week4asgmt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [1]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [ ]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
n_missing_study = students["study_hours_reported"].isna().sum()

**Your justification (1–2 sentences):** _..._

I used Han’s global-mean imputation method. Observed study hours are approximately symmetric (mean 10.49, median 10.5, skewness about −0.07), so the mean is a reasonable single replacement. This retains all 2,027 rows, though imputed values reduce the apparent variation in study hours.

In [ ]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [2]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups

print("Before cleaning:")
print(students["housing"].value_counts())

housing_key = (
    students["housing"]
    .str.strip()
    .str.lower()
    .str.replace("-", " ", regex=False)
)

students["housing_clean"] = housing_key.map({
    "on campus": "On-Campus",
    "off campus": "Off-Campus",
    "with family": "With Family",
})

print("\nAfter cleaning:")
print(students["housing_clean"].value_counts())


Before cleaning:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

After cleaning:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [3]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [4]:
# TODO
bad_age = ~students["age"].between(16, 110)
bad_work = students["work_hours_per_week"] < 0

impossible_ages = sorted(students.loc[bad_age, "age"].unique().tolist())
n_neg_work = int(bad_work.sum())

print("Impossible ages:", impossible_ages)
print("Rows with negative work hours:", n_neg_work)

Impossible ages: [-22, 0, 1, 3, 199, 220]
Rows with negative work hours: 4


In [5]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [6]:
# TODO: build students_dedup (one row per student)
print("Roster rows before:", len(students))
print("Repeated student IDs:", students["student_id"].duplicated().sum())

students_dedup = students.drop_duplicates(
    subset="student_id",
    keep="first"
).copy()

print("Roster rows after:", len(students_dedup))

Roster rows before: 2027
Repeated student IDs: 27
Roster rows after: 2000


**Why not de-dupe enroll / activity? (1 sentence):** _..._

Each enrollment row represents a course and each activity row represents a week, so the same student ID should appear multiple times in those files.

In [7]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [8]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
# Give all three files the same kind of ID.
students_dedup["sid"] = students_dedup["student_id"].str.replace("NU-", "").astype(int)
activity["sid"] = activity["student_id"].str.replace("NU-", "").astype(int)

# Count courses and add up activity minutes for each student.
course_counts = enroll.groupby("sid").size().reset_index(name="course_count")
activity_totals = activity.groupby("sid")["minutes_active"].sum().reset_index()

# Start with the student roster and attach those summaries.
analysis = students_dedup.merge(course_counts, on="sid", how="left")
analysis = analysis.merge(activity_totals, on="sid", how="left")

print(len(analysis))

2000


In [9]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [10]:
# TODO
is_match = enroll["sid"].isin(students_dedup["sid"])

matched = is_match.sum()
orphan_rows = (~is_match).sum()
orphan_ids = enroll.loc[~is_match, "sid"].nunique()

print("Matched enrollment rows:", matched)
print("Orphan enrollment rows:", orphan_rows)
print("Different orphan IDs:", orphan_ids)
print("Analysis rows:", len(analysis))
print("Cleaned roster rows:", len(students_dedup))


Matched enrollment rows: 8041
Orphan enrollment rows: 47
Different orphan IDs: 14
Analysis rows: 2000
Cleaned roster rows: 2000


In [11]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [12]:
# TODO
blanks_before = analysis["enrollment_date"].isna().sum()

dates_parsed = pd.to_datetime(
    analysis["enrollment_date"],
    format="mixed",
    errors="coerce"
)

blanks_after = dates_parsed.isna().sum()
analysis["enrollment_date"] = dates_parsed

print("Blanks before:", blanks_before)
print("Blanks after:", blanks_after)

Blanks before: 0
Blanks after: 0


In [13]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [14]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
hours = analysis["study_hours_reported"]

analysis["study_z"] = (hours - hours.mean()) / hours.std()

analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[-0.01, 1, 2, 3, 4],
    labels=["0–1", ">1–2", ">2–3", ">3–4"]
)

print("Average z-score:", analysis["study_z"].mean())
print(analysis["gpa_band"].value_counts())

Average z-score: 4.3442403804347476e-16
gpa_band
>2–3    1102
>1–2     528
>3–4     300
0–1       70
Name: count, dtype: int64


In [15]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [16]:
# TODO: write analysis to northgate_clean.csv
analysis.to_csv("northgate_clean.csv", index=False)

In [17]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*

**Your cleaning log:**

- Missing study hours: Filled 255 blanks with the column mean. The observed values were approximately symmetric, and this kept the sample size unchanged.
- Housing: Standardized spaces, capitalization, and hyphens so eight spellings became three housing groups.
- Impossible values: Marked implausible ages and negative work hours as missing because their correct values cannot be determined. Kept extreme but possible values.
- Duplicate students: Kept one row per student ID, reducing the roster from 2,027 rows to 2,000.
- Integration: Removed `NU-` from the IDs, summarized courses and weekly minutes by student, and joined those summaries to the roster. Enrollment IDs absent from the roster were excluded from the analysis table.
- Dates: Parsed the mixed date formats and confirmed that the number of blank dates remained zero.
- New columns: Converted study hours to z-scores and divided final GPA into four bands.

### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

In [18]:
# TODO: compute the mean GPA and explore one relationship on the CLEAN data
print("Clean mean GPA:", round(analysis["final_gpa"].mean(), 3))
print("Raw mean GPA:", round(students["final_gpa"].mean(), 3))

print("\nDropout rate by housing:")
print(
    (analysis.groupby("housing_clean")["dropped_out"].mean() * 100)
    .round(1)
)

Clean mean GPA: 2.305
Raw mean GPA: 2.307

Dropout rate by housing:
housing_clean
Off-Campus     22.1
On-Campus      19.2
With Family    21.5
Name: dropped_out, dtype: float64


The cleaned mean GPA is 2.305. Dropout rates are slightly lower for On-Campus students (19.2%) than for students living With Family (21.5%) or Off-Campus (22.1%). Cleaning combined inconsistent housing labels and removed repeated student records, while changing the overall mean GPA only slightly from 2.307 to 2.305.

In [19]:
# ✅ Check
print("(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)")

(D11 is interpreted by your instructor — make sure your numbers and one-sentence takeaway are shown above.)
